# 02 — WBGT Processing: Daily WBGT → Annual Productivity Loss Rasters

For each year in the configured range, this notebook:
1. Streams daily WBGT GeoTIFFs directly from CHIRTS-ERA5 via `/vsicurl/` (no files saved to disk)
2. Applies the exposure-response function (WBGT → productivity loss)
3. Averages across all days in the year
4. Saves one **mean annual productivity loss** raster per year to `data/processed/annual/`

Saving per-year rasters lets you freely define epochs in later notebooks by averaging any subset of years.

**Before running:** ensure `scripts/productivity.py` has `WBGT_THRESHOLDS` and `PRODUCTIVITY_VALUES` defined.

In [ ]:
import sys
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import rasterio
import yaml
from tqdm import tqdm

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from scripts.productivity import wbgt_to_productivity_loss

with open(ROOT / 'config' / 'config.yaml') as f:
    config = yaml.safe_load(f)

ANNUAL_DIR = ROOT / 'data' / 'processed' / 'annual'
ANNUAL_DIR.mkdir(parents=True, exist_ok=True)

URL_TEMPLATE = config['wbgt']['url_template']
print('Ready. Output dir:', ANNUAL_DIR)

## Configuration

In [ ]:
# Year range to process — adjust as needed
START_YEAR = config['wbgt']['start_year']
END_YEAR   = config['wbgt']['end_year']

# Skip years that already have an output raster
SKIP_EXISTING = True

years_to_process = [
    y for y in range(START_YEAR, END_YEAR + 1)
    if not SKIP_EXISTING or not (ANNUAL_DIR / f'productivity_loss_{y}.tif').exists()
]

print(f'Years to process: {len(years_to_process)} ({START_YEAR}–{END_YEAR})')
if years_to_process:
    print(f'First: {years_to_process[0]}, Last: {years_to_process[-1]}')

## Process one raster per year

In [ ]:
def iter_dates(year: int):
    """Yield every date in a given year."""
    d = date(year, 1, 1)
    while d.year == year:
        yield d
        d += timedelta(days=1)


def process_year(year: int) -> tuple[np.ndarray, dict] | None:
    """
    Stream all daily WBGT files for a year via vsicurl, apply the ERF,
    and return the mean annual productivity loss array and raster metadata.
    Returns None if no valid data was found.
    """
    accumulator = None
    valid_count = None
    ref_meta = None
    nodata_val = None
    skipped = 0

    for d in iter_dates(year):
        url = '/vsicurl/' + URL_TEMPLATE.format(year=d.year, month=d.month, day=d.day)
        try:
            with rasterio.open(url) as src:
                if ref_meta is None:
                    ref_meta = src.meta.copy()
                    ref_meta.update(dtype='float32', nodata=np.nan)
                    shape = (src.height, src.width)
                    nodata_val = src.nodata
                    accumulator = np.zeros(shape, dtype=np.float64)
                    valid_count = np.zeros(shape, dtype=np.int32)

                wbgt = src.read(1).astype(np.float32)

        except Exception:
            skipped += 1
            continue

        if nodata_val is not None:
            wbgt[wbgt == nodata_val] = np.nan

        loss = wbgt_to_productivity_loss(wbgt)
        valid = ~np.isnan(loss)
        accumulator[valid] += loss[valid]
        valid_count[valid] += 1

    if accumulator is None or valid_count.max() == 0:
        return None

    if skipped:
        print(f'  {year}: {skipped} days skipped (file unavailable)')

    mean_loss = np.where(valid_count > 0, accumulator / valid_count, np.nan).astype(np.float32)
    return mean_loss, ref_meta


for year in tqdm(years_to_process, desc='Years'):
    out_path = ANNUAL_DIR / f'productivity_loss_{year}.tif'
    result = process_year(year)
    if result is None:
        print(f'  {year}: no data — skipping')
        continue
    mean_loss, ref_meta = result
    with rasterio.open(out_path, 'w', **ref_meta) as dst:
        dst.write(mean_loss, 1)

print('Done.')

## Check available annual rasters

In [ ]:
annual_files = sorted(ANNUAL_DIR.glob('productivity_loss_*.tif'))
years_available = [int(f.stem.split('_')[-1]) for f in annual_files]
print(f'Annual rasters available: {len(annual_files)}')
if years_available:
    print(f'Range: {min(years_available)}–{max(years_available)}')

## Quick visual check — single year

In [ ]:
import matplotlib.pyplot as plt

CHECK_YEAR = years_available[-1] if years_available else None

if CHECK_YEAR:
    with rasterio.open(ANNUAL_DIR / f'productivity_loss_{CHECK_YEAR}.tif') as src:
        arr = src.read(1)

    print(f'Year: {CHECK_YEAR}')
    print(f'Loss range: {np.nanmin(arr):.4f} – {np.nanmax(arr):.4f}')
    print(f'Mean (valid cells): {np.nanmean(arr):.4f}')

    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(arr, cmap='YlOrRd', vmin=0, vmax=np.nanmax(arr))
    plt.colorbar(im, ax=ax, label='Mean annual productivity loss (fraction)')
    ax.set_title(f'Average Daily Productivity Loss — {CHECK_YEAR}')
    plt.tight_layout()
    plt.show()